<a href="https://colab.research.google.com/github/broadinstitute/BE3D/blob/main/examples/BE3Dv6_MultipleScreens_LocalScriptNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# README
- This is an example for MEN1 which is based on two CBE screens


# Setup

In [ ]:
# @title Install DSSP and ClustalO

! apt-get update
! apt-get install dssp clustalo


In [ ]:
# @title Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title Install BE3D

! pip install git+https://github.com/broadinstitute/beclust3d-public.git
print('beclust3d installed at:')
! pip show beclust3d


In [ ]:
# @title Download relevant files

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/PernerNature2023-MOLM13-Screen.tsv -O PernerNature2023-MOLM13-Screen.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/PernerNature2023-MV411-Screen.tsv -O PernerNature2023-MV411-Screen.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/men1.fasta -O men1.fasta
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/pdb/men1.pdb -O men1.pdb

! mkdir temp
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/yaml/men1.yaml -O temp/men1.yaml

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_local.py -O be3d_local.py


In [ ]:
# @title Import packages

import os
import sys
import yaml
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import gc
import uuid
import ipywidgets as widgets
from IPython.display import display

from IPython.display import Image, display, SVG
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from google.colab import files
import shutil

from beclust3d import *

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)


# Input (optional)
This section creates a .yaml file, so this step is not necessary if you use the .yaml file from the Github as is

In [ ]:
# @title Required (filled with default values for MEN1)

yaml_dict = {}

beclust3d_path = '../'  # @param {type:"string"}
yaml_dict['beclust3d_path'] = beclust3d_path

screen_dir = '/content/'  # @param {type:"string"}
yaml_dict['screen_dir'] = screen_dir

nRandom = 500  # @param {type:"integer"}
yaml_dict['nRandom'] = nRandom


# ---- pthr ----
yaml_dict['pthr'] = {}

single_screen = 0.05  # @param {type:"number"}
yaml_dict['pthr']['single_screen'] = single_screen

multi_screen = 0.05  # @param {type:"number"}
yaml_dict['pthr']['multi_screen'] = multi_screen


structure_radius = 6.0  # @param {type:"number"}
yaml_dict['structure_radius'] = structure_radius

clustering_radius = 6.0  # @param {type:"number"}
yaml_dict['clustering_radius'] = clustering_radius

function_for_lfc = 'max'  # @param {type:"string"}
yaml_dict['function_for_lfc'] = function_for_lfc

function_for_lfc3d = 'mean'  # @param {type:"string"}
yaml_dict['function_for_lfc3d'] = function_for_lfc3d

function_for_meta = 'SUM'  # @param {type:"string"}
yaml_dict['function_for_meta'] = function_for_meta


# ---- database ----
yaml_dict['database'] = {}

mut_list_col = None  # @param {type:"string"}
yaml_dict['database']['mut_list_col'] = mut_list_col

mut_col = 'mutation_category'  # @param {type:"string"}
yaml_dict['database']['mut_col'] = mut_col

val_col = 'delta_beta_score'  # @param {type:"string"}
yaml_dict['database']['val_col'] = val_col

gene_col = 'Gene Symbol'  # @param {type:"string"}
yaml_dict['database']['gene_col'] = gene_col

edits_col = 'predicted_edit'  # @param {type:"string"}
yaml_dict['database']['edits_col'] = edits_col

mut_delimiter = ';'  # @param {type:"string"}
yaml_dict['database']['mut_delimiter'] = mut_delimiter

gRNA_col = None  # @param {type:"string"}
yaml_dict['database']['gRNA_col'] = gRNA_col


# ---- conservation ----
yaml_dict['conservation'] = {}

conservation_run = False  # @param {type:"boolean"}
yaml_dict['conservation']['run'] = conservation_run

v_score_threshold = 3  # @param {type:"integer"}
yaml_dict['conservation']['v_score_threshold'] = v_score_threshold

alt_gene_name = None  # @param {type:"string"}
yaml_dict['conservation']['alt_gene_name'] = alt_gene_name

alt_uniprot_id = None  # @param {type:"string"}
yaml_dict['conservation']['alt_uniprot_id'] = alt_uniprot_id

alt_screen_start = None  # @param {type:"string"}
yaml_dict['conservation']['alt_screen_start'] = alt_screen_start

muscle_path = 'muscle'  # @param {type:"string"}
yaml_dict['muscle_path'] = muscle_path

# ---- mutation_category ----
yaml_dict['mutation_category'] = {}

missense = ['Missense']  # @param {type:"raw"}
yaml_dict['mutation_category']['missense'] = missense

silent = ['Silent']  # @param {type:"raw"}
yaml_dict['mutation_category']['silent'] = silent

nonsense = ['Nonsense']  # @param {type:"raw"}
yaml_dict['mutation_category']['nonsense'] = nonsense

no_mutation = ['No Mutation']  # @param {type:"raw"}
yaml_dict['mutation_category']['no_mutation'] = no_mutation

splice = ['Splice']  # @param {type:"raw"}
yaml_dict['mutation_category']['splice'] = splice

intron = ['Intron']  # @param {type:"raw"}
yaml_dict['mutation_category']['intron'] = intron

# ---- qa ----
yaml_dict['qa'] = {}

qa_passed_only = False  # @param {type:"boolean"}
yaml_dict['qa']['qa_passed_only'] = qa_passed_only

qa_only = False  # @param {type:"boolean"}
yaml_dict['qa']['qa_only'] = qa_only

cases = ['Nonsense', 'Splice']  # @param {type:"raw"}
yaml_dict['qa']['cases'] = cases

controls = ['No Mutation']  # @param {type:"raw"}
yaml_dict['qa']['controls'] = controls

input_gene = 'MEN1'  # @param {type:"string"}
yaml_dict['input_gene'] = input_gene

input_uniprot = 'O00255'  # @param {type:"string"}
yaml_dict['input_uniprot'] = input_uniprot

input_chain = 'A'  # @param {type:"string"}
yaml_dict['input_chain'] = input_chain

screens = 'PernerNature2023-MOLM13-Screen.tsv, PernerNature2023-MV411-Screen.tsv'  # @param {type:"string"}
yaml_dict['screens'] = screens

output_dir = '/content/'  # @param {type:"string"}
yaml_dict['output_dir'] = output_dir

user_fasta = '/content/men1.fasta'  # @param {type:"string"}
yaml_dict['user_fasta'] = user_fasta

user_pdb = '/content/men1.pdb'  # @param {type:"string"}
yaml_dict['user_pdb'] = user_pdb

user_dssp = None  # @param {type:"string"}
yaml_dict['user_dssp'] = user_dssp

priority_on_alternative = False  # @param {type:"boolean"}
yaml_dict['priority_on_alternative'] = priority_on_alternative

ppi_chain_gene_dict = None  # @param {type:"raw"}
yaml_dict['ppi_chain_gene_dict'] = ppi_chain_gene_dict

ppi_gene_edits_dict = None  # @param {type:"raw"}
yaml_dict['ppi_gene_edits_dict'] = ppi_gene_edits_dict

atom_level_naa = False  # @param {type:"boolean"}
yaml_dict['atom_level_naa'] = atom_level_naa


In [ ]:
# @title Convert dictionary to yaml file

import yaml

yaml_filename = 'temp/men1.yaml' # @param {type:"string"}

with open(yaml_filename, 'w') as file:
    yaml.dump(yaml_dict, file, sort_keys=False, default_flow_style=False)

print(f"YAML file '{yaml_filename}' created successfully.")


# Running BE3D

In [ ]:
# Run script

! python be3d_local.py temp/men1.yaml


In [ ]:
# @title Download output directory

yaml_filename = 'temp/men1.yaml' # @param {type:"string"}
with open(yaml_filename, 'r') as file:
    input = yaml.safe_load(file)
    output_dir = input['output_dir']

if output_dir == '/content/':
    print(f'Downloading {output_dir} is not recommended. Try downloading specific folders or changing [output_dir].')

else:
    download_directory = True #@param {type:"boolean"}
    if download_directory:
        shutil.make_archive(output_dir, 'zip', output_dir)
        files.download(f"{output_dir}.zip")
